In [2]:
import os
import glob
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import emoji
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import html
import requests
from datasets import Dataset, DatasetDict
from datetime import datetime, timedelta
from hmmlearn import hmm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
from multiprocessing.sharedctypes import Value
from indobert import pattern
data_folder = "/Users/asyzyni/Desktop/TA /Kode dan Eksperimen/Kode TA/Percobaan I /data"
output_folder = "./cleaned_data"

os.makedirs(output_folder, exist_ok=True)

csv_files = glob.glob(os.path.join(data_folder, "*.csv"))

failed_files = []
success_files = []

for filepath in csv_files:
    filename = os.path.basename(filepath)

    match = re.search(r"(.+?)\.csv", filename)

    if match:
        product_id = match.group(1).strip()
    else:
        product_id = filename.replace('.csv', '').strip()
    
    try: 
        df = pd.read_csv(filepath, dtype=str, keep_default_na=False)
        df['product_id'] = product_id 

        metadata_col = ['web_scraper_order', 'web_scraper_start_url', 'pagination', 
                        'product_id', 'timestamp', 'phone', 'data']
        review_col = []

        if 'review' in df.columns: 
            review_col = ['review']
        else: 
            for col in df.columns: 
                if col not in metadata_col: 
                    col_lower = col.lower()
                    if re.match(r'^data\d+$', col_lower):
                        review_col.append(col)
                    elif any(pattern in col_lower for pattern in ['review_text', 'review_content', 'content', 'text', 'comment', 'ulasan', 'body', 'description']):
                        review_col.append(col)
        
        # jika tidak ditemukan cari kolom teks panjang
        if not review_col:
            for col in df.columns:
                if col not in metadata_col: 
                    sample = df[col].dropna().head(10)
                    if len(sample) > 0: 
                        avg_len = sample.astype(str).str.len().mean()
                        if avg_len > 20: 
                            review_col.append(col)

        # gabungkan kolom review  
        if len(review_col) > 0: 
            def safe_join(row): 
                parts = []
                for col in review_col:
                    val = row[col]
                    if pd.isna(val) or val is None: 
                        continue 
                    val_str = str(val).strip()

                    if val_str and val_str.lower() not in ['nan', '', 'none']:
                        if len(val_str) > 1:
                            parts.append(val_str)
                
                result = ' '.join(parts).strip()
                result = re.sub(r'\s+', ' ', result)
                return result 
            
            df['review'] = df.apply(safe_join, axis=1)
        
        else: 
            raise ValueError("Tidak menemukan kolom review")
        
        timestamp_col = None

        if 'timestamp' in df.columns:
            timestamp_col = 'timestamp'
        else:
            for col in df.columns:
                col_lower = col.lower()

                if any(pattern in col_lower for pattern in ['date', 'created_at', 'time', 'datetime', 'tanggal']):
                    if col not in review_col and col not in metadata_col:
                        timestamp_col = col 
                        break
        
        if timestamp_col:
            df['timestamp_clean'] = df[timestamp_col].astype(str).str.split('|').str[0].str.strip()
            try:
                df['timestamp'] = pd.to_datetime(df['timestamp_clean'], errors='coerce')
                df = df.dropna(subset=['timestamp'])
            
            df = df.drop(columns=['timestamp_clean'], errors='ignore')
        else: 
            df['timestamp'] = None 




In [ ]:
a = pd.read_csv("")